# Multi-Head Latent Attention（MLA）

源码导航：[core/attention/mla.py](../../../core/attention/mla.py) 中的 `MultiHeadLatentAttention`。

DeepSeek-V2 提出的 MLA 将 K/V **联合压缩到低秩潜空间**，推理时 KV cache 只需存储潜向量（维度 `kv_lora_rank`），而非完整的 per-head K/V。与 MQA/GQA 减少 KV 头数不同，MLA 在**保持多头表达能力**的同时压缩 cache。

### 1. 理论推导

1. **KV 压缩**：$c_{kv} = \text{RMSNorm}(x W_{kv\downarrow})$，再 $[K_{nope}; V] = c_{kv} W_{kv\uparrow}$
2. **RoPE 解耦**：位置信息存入独立的 `qk_rope_head_dim` 维，与压缩内容维分离
3. **Q 低秩**（可选）：$c_q = \text{RMSNorm}(x W_{q\downarrow}) \to Q = c_q W_{q\uparrow}$

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.attention.mla import MultiHeadLatentAttention

### 2. Forward 形状检查

In [ ]:
torch.manual_seed(0)
B, T, C = 2, 16, 128
H = 4
x = torch.randn(B, T, C)

mla = MultiHeadLatentAttention(
    n_embd=C, n_head=H, kv_lora_rank=32,
    qk_nope_head_dim=32, qk_rope_head_dim=16, v_head_dim=32,
    attn_impl='eager',
)
y = mla(x)
assert y.shape == (B, T, C)
print('output:', tuple(y.shape))
print('KV cache per token:', mla.kv_cache_size_per_token())
print('MHA equivalent:', 2 * H * (32 + 16))

### 3. KV cache 对比表

In [ ]:
H, d, rank = 16, 64, 512
print(f'MHA:  {2*H*d}')
print(f'GQA (4 KV): {2*4*d}')
print(f'MQA:  {2*d}')
print(f'MLA (rank={rank}): {rank + 32}  # latent + rope part')

---

## 延伸阅读与参考资料

- DeepSeek-AI, *DeepSeek-V2* (2024). [arXiv:2405.04434](https://arxiv.org/abs/2405.04434)
- Weight absorption：生产推理中将 up-proj 融入 Q 计算，本教学实现未包含。